# Multimodal Airline AI Assistant

Build an airline customer-support assistant that can look up ticket prices, generate destination images, and speak its responses. The project combines OpenAI tool calling, SQLite, image generation, text-to-speech, and a custom Gradio interface.

**Features**
- Look up return-ticket prices from a SQLite database
- Use an OpenAI function tool to retrieve pricing data
- Generate a destination image after a price lookup
- Convert the assistant's response to speech
- Present chat, image, and audio outputs in a Gradio `Blocks` interface

> Image and audio generation use paid API calls. Run those cells only when you intend to incur usage costs.

## 1. Install dependencies

```bash
pip install openai gradio python-dotenv pillow
```

Create a `.env` file beside the notebook:

```text
OPENAI_API_KEY=your_api_key
OPENAI_MODEL=gpt-4.1-mini
```

The model name can be changed to another model available to your API account.

In [ ]:
import json
import os
import sqlite3
import tempfile

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
from io import BytesIO
import base64

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your .env file before running this notebook.")

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
DB_PATH = "prices.db"

SYSTEM_MESSAGE = (
    "You are a helpful customer-support assistant for an airline called FlightAI. "
    "Give short, courteous, accurate answers. Use the ticket-price tool when a customer "
    "asks about a destination's return-ticket price. If the information is unavailable, say so."
)


## 2. Create and seed the ticket database

This makes the notebook self-contained. SQLite stores destination cities and their return-ticket prices.

In [ ]:
def initialise_database():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS prices (
                city TEXT PRIMARY KEY,
                price REAL NOT NULL
            )
            """
        )

        sample_prices = [
            ("paris", 899),
            ("london", 799),
            ("tokyo", 1420),
            ("sydney", 2999),
            ("new york city", 1100),
        ]
        connection.executemany(
            "INSERT OR IGNORE INTO prices (city, price) VALUES (?, ?)",
            sample_prices,
        )

def get_ticket_price(city):
    with sqlite3.connect(DB_PATH) as connection:
        row = connection.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city.strip().lower(),),
        ).fetchone()

    if row:
        return f"The return-ticket price to {city.title()} is ${row[0]:,.2f}."
    return f"No ticket-price data is currently available for {city.title()}."

initialise_database()
print(get_ticket_price("Paris"))


## 3. Define the ticket-price tool

The tool schema tells the model which function it can request and which argument it must provide. The model does not access SQLite directly; the Python application executes the function.

In [ ]:
price_tool = {
    "type": "function",
    "function": {
        "name": "get_ticket_price",
        "description": "Get the price of a return ticket to a destination city.",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string",
                    "description": "The destination city the customer wants to visit.",
                }
            },
            "required": ["destination_city"],
            "additionalProperties": False,
        },
    },
}

tools = [price_tool]


## 4. Handle tool calls

When the model requests the ticket-price function, parse its JSON arguments, run the Python function, and return the result with the matching tool-call ID.

In [ ]:
def handle_tool_calls(assistant_message):
    tool_responses = []
    requested_cities = []

    for tool_call in assistant_message.tool_calls or []:
        if tool_call.function.name != "get_ticket_price":
            continue

        arguments = json.loads(tool_call.function.arguments)
        city = arguments["destination_city"]
        requested_cities.append(city)

        tool_responses.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": get_ticket_price(city),
        })

    return tool_responses, requested_cities


## 5. Generate a destination image

The image-generation function creates a destination illustration. It is called after the assistant uses the ticket-price tool.

This API call may incur a cost.

In [ ]:
def generate_destination_image(city):
    result = client.images.generate(
        model="gpt-image-1-mini",
        prompt=(
            f"Create a vibrant pop-art travel poster representing a vacation in {city}. "
            "Show recognisable landmarks and distinctive local scenery. "
            "Do not include text or lettering."
        ),
        size="1024x1024",
        n=1,
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)
    return Image.open(BytesIO(image_bytes))


## 6. Convert the response to speech

Text-to-speech generates an audio version of the assistant's reply. The function saves the generated MP3 to a temporary file for Gradio to play.

In [ ]:
def generate_speech(message):
    speech = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=message,
        response_format="mp3",
    )

    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as audio_file:
        audio_path = audio_file.name

    speech.stream_to_file(audio_path)
    return audio_path


## 7. Build the multimodal chat workflow

The workflow:
1. Adds the latest user message to the conversation.
2. Sends the conversation and tool schema to the model.
3. Executes requested tool calls and returns their results.
4. Repeats until the model returns a final text response.
5. Generates speech for the response.
6. Generates a destination image if a ticket-price lookup was performed.

In [ ]:
def respond(history):
    messages = [{"role": "system", "content": SYSTEM_MESSAGE}] + history
    requested_cities = []

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
    )

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses, cities = handle_tool_calls(assistant_message)
        requested_cities.extend(cities)

        messages.append(assistant_message.model_dump(exclude_none=True))
        messages.extend(tool_responses)

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

    reply = response.choices[0].message.content or "I'm sorry, I couldn't generate a response."
    return reply, requested_cities


## 8. Create the Gradio interface

Gradio `Blocks` allows the chat, image, and audio components to be arranged in one custom interface. The callback updates all three outputs.

In [ ]:
def chat_with_assistant(message, history):
    history = history or []
    history = history + [{"role": "user", "content": message}]

    reply, cities = respond(history)
    history = history + [{"role": "assistant", "content": reply}]

    audio_path = generate_speech(reply)
    destination_image = generate_destination_image(cities[0]) if cities else None

    return history, "", audio_path, destination_image


with gr.Blocks(title="FlightAI Multimodal Assistant") as app:
    gr.Markdown("# FlightAI Multimodal Assistant")
    gr.Markdown("Ask about return-ticket prices and receive text, audio, and destination imagery.")

    chatbot = gr.Chatbot(type="messages", height=420)
    with gr.Row():
        message_box = gr.Textbox(
            label="Your message",
            placeholder="What is the return-ticket price to Paris?",
            scale=5,
        )
        send_button = gr.Button("Send", variant="primary", scale=1)

    with gr.Row():
        audio_output = gr.Audio(label="Spoken response", autoplay=True)
        image_output = gr.Image(label="Destination image")

    send_button.click(
        chat_with_assistant,
        inputs=[message_box, chatbot],
        outputs=[chatbot, message_box, audio_output, image_output],
    )
    message_box.submit(
        chat_with_assistant,
        inputs=[message_box, chatbot],
        outputs=[chatbot, message_box, audio_output, image_output],
    )

app.launch()


## 9. Gradio interface options

Gradio provides different ways to build interfaces:

- **`gr.Interface`:** A straightforward interface for a function with inputs and outputs.
- **`gr.ChatInterface`:** A ready-made conversational interface.
- **`gr.Blocks`:** A flexible layout for custom interfaces with multiple components and event callbacks.

This project uses `gr.Blocks` because it needs to display chat, image, and audio outputs together.

## Key concepts

- **Tool calling:** The model requests that the application run a named function.
- **Database integration:** Python retrieves ticket prices from SQLite.
- **Multimodal output:** The application returns text, generated images, and generated speech.
- **Text-to-image generation:** The image API creates a visual representation of a destination.
- **Text-to-speech:** The audio API converts the assistant's response into spoken audio.
- **Gradio callbacks:** UI events connect user input to the Python workflow.
- **Agentic workflow:** The model can request external actions through tools, while the application controls execution.

## Business applications

The same pattern can be adapted for customer support, travel planning, product assistance, and internal business workflows. Real booking or payment operations should include authentication, input validation, and explicit confirmation before making changes.

## Further exercises

- Add a flight-search tool.
- Add a booking lookup tool.
- Add a confirmation step before creating or changing a booking.
- Add error handling for unavailable destinations and failed API requests.
- Add speech input so users can talk to the assistant.